In [3]:
import numpy as np
import pandas as pd
import plotly.express as px
import torchvision
import os
from scipy.ndimage import rotate
import random
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
import plotly.graph_objects as go
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import confusion_matrix
from sklearn.linear_model import LinearRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
import joblib  # For saving/loading the model

pd.options.plotting.backend = "plotly"

In [4]:
# Set random seeds for reproducible results
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
print(3 / 2)

# IMPORTANT: set save_models to True to save trained models.
save_models = True
load_saved_models = True # After training, you can set this to True to load the saved models and not have to re-train them.

1.5


In [5]:
# Importing helper function to show images
def show_images(images, max_images=40, ncols=5, labels = None, reshape=False):
    """Visualize a subset of images from the dataset.
    Args:
        images (np.ndarray or list): Array of images to visualize [img,row,col].
        max_images (int): Maximum number of images to display.
        ncols (int): Number of columns in the grid.
        labels (np.ndarray, optional): Labels for the images, used for facet titles.
    Returns:
        plotly.graph_objects.Figure: A Plotly figure object containing the images.
    """
    if isinstance(images, list):
        images = np.stack(images)
    n = min(images.shape[0], max_images) # number of images to show
    px_height = 220 # height of each image in pixels
    if reshape:
        images = images.reshape(images.shape[0], 28, 28)
    fig = px.imshow(images[:n, :, :], color_continuous_scale='gray_r', 
                    facet_col = 0, facet_col_wrap=ncols,
                    height = px_height * int(np.ceil(n/ncols)))
    fig.update_layout(coloraxis_showscale=False)
    fig.update_xaxes(showticklabels=False, showgrid=False)
    fig.update_yaxes(showticklabels=False, showgrid=False)
    if labels is not None:
        # Extract the facet number and replace with the label.
        fig.for_each_annotation(lambda a: a.update(text=labels[int(a.text.split("=")[-1])]))
    return fig

## Load the Fashion-MNIST Dataset  

Let's reload the Fashion-MNIST dataset. As a reminder, this dataset contains 70k images (60k training, 10k testing) of 28x28 grayscale images of clothing divided into the following 10 classes:
1. **T-shirt/top**
2. **Trouser**
3. **Pullover**
4. **Dress**
5. **Coat**
6. **Sandal**
7. **Shirt**
8. **Sneaker**
9. **Bag**
10. **Ankle boot**

### Dataset Structure:
- **Images**: Each image is represented as a 28x28 grayscale matrix, where each value corresponds to the pixel intensity.
- **Targets**: Each image is associated with a label (0-9), which maps to one of the 10 classes listed above.

### Data Preparation:
In the code cell below, we:
1. Load the Fashion-MNIST training dataset using `torchvision.datasets.FashionMNIST`.
2. Extract the images and targets into `NumPy` arrays for easier manipulation.
3. Create a `pandas DataFrame` (`df`) to store the images and their corresponding labels. 

In [6]:
# Load the FashionMNIST training dataset
train_data = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True)
images = train_data.data.numpy().astype(float)  # Convert images to numpy array
targets = train_data.targets.numpy()  # Extract labels
class_dict = {i:class_name for i,class_name in enumerate(train_data.classes)}
labels = np.array([class_dict[t] for t in targets])  # Map labels to class names
n = len(images)  # Total samples
class_names = list(class_dict.values())
print(f"Loaded FashionMNIST with {n} samples. Classes: {class_dict}")
print("Classes: {}".format(class_dict))
print("Image shape: {}".format(images[0].shape))
print("Image dtype: {}".format(images[0].dtype))


# Creating a DataFrame with the images and labels
df = pd.DataFrame({"image": images.tolist(), "label": labels})
# Cast image as numpy array
df['image'] = df['image'].apply(lambda x: np.array(x).reshape(-1))

Loaded FashionMNIST with 60000 samples. Classes: {0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat', 5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'}
Classes: {0: 'T-shirt/top', 1: 'Trouser', 2: 'Pullover', 3: 'Dress', 4: 'Coat', 5: 'Sandal', 6: 'Shirt', 7: 'Sneaker', 8: 'Bag', 9: 'Ankle boot'}
Image shape: (28, 28)
Image dtype: float64


In [7]:
# Split the dataset into training and testing sets (80% train, 20% test)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=SEED)

# Extract image data and labels for training and testing
X_train = np.stack(train_df['image'].to_numpy())  # Training images
X_test = np.stack(test_df['image'].to_numpy())    # Testing images
y_train = train_df['label'].to_numpy()            # Training labels
y_test = test_df['label'].to_numpy()              # Testing labels

# Print the shapes of the training and testing datasets
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

X_train shape: (48000, 784)
X_test shape: (12000, 784)


## Даалгавар 6: Regression Analysis

The goal is to predict the price of clothing items based on their image features. Follow the steps below to load the price data, train a regression model, and evaluate its performance.

### Problem 6a: Training a Linear Regression Model

**Task**: Use `pandas` and `scikit-learn` to train a linear regression model to predict prices:

1. **Join the DataFrames**: 
    - Combine the price data with the original `train_df` and `test_df` using the [`join`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.join.html) method.
    - Store the resulting training data in `prices_train` and the testing data in `prices_test`.

    **Hint**: Think about which column can serve as the key for joining the `prices` DataFrame with the original `train_df` and `test_df`.

2. **Train the Model**:
    - Use `LinearRegression` from [`sklearn.linear_model`](https://scikit-learn.org/stable/modules/linear_model.html) to train a regression model.
    - Ensure that the model predicts only positive prices.

3. **Store Predictions**:
    - Add the predicted prices to the `prices_test` `DataFrame`.

**Important for grading**: Name your model `price_model`.

**Expected Output**:
The `prices_train` and `prices_test` `DataFrames` should look like this:

| key_0 |      image      |    label    | predicted_label | correct | Price |
|-------|-----------------|-------------|-----------------|---------|-------|
| 48572 | [0.0, 0.0, ...] |   Sneaker   |     Sneaker     |   True  | 49.86 |
| 38696 | [0.0, 0.0, ...] |    Dress    |      Dress      |   True  | 14.56 |

In [8]:
prices = pd.read_csv("FashionMNIST_prices.csv")
# TODO: Join the original `train_df` and `test_df` with the `prices` DataFrame
prices_train = train_df.copy()
prices_test = test_df.copy()

prices_train["Price"] = prices.loc[train_df.index, "Price"].values
prices_test["Price"] = prices.loc[test_df.index, "Price"].values

prices_train['Price'].hist()
plt.show()
prices_train.head()

,image,label,Price
48572,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Sneaker,49.86
38696,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Dress,14.56
13611,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Sandal,211.99
35213,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ...",Bag,498.13
31766,"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 3.0, 57.0,...",Shirt,82.48


In [11]:
# TODO: Fit a linear regression model (price_model) on the training data with prices as the target variable
if load_saved_models and os.path.exists('price_model.joblib'):
    price_model = joblib.load('price_model.joblib')
    print("Model loaded from price_model.joblib")
else:
    price_model = LinearRegression()
    price_model.fit(X_train, prices_train["Price"])
    print("Model trained")

# TODO: Predict the prices of the test data and add them to the test_df
prices_test['price_prediction'] = price_model.predict(X_test)

if save_models:
    joblib.dump(price_model, 'price_model.joblib')
    print("Model saved to price_model.joblib")

Model loaded from price_model.joblib
Model saved to price_model.joblib


### Problem 6b: Evaluate Linear Regression Performance

**Task**: Use `NumPy` operations to evaluate the performance of the linear regression model:

1. Calculate the **Root Mean Squared Error (RMSE)** to measure the average magnitude of prediction errors.

2. Calculate the **Mean Absolute Error (MAE)** to measure the average absolute difference between predicted and actual values.

3. Calculate the **R-squared score (R²)** to assess how well the model explains the variance in the data.

In [10]:
def compute_rmse(y_true, y_pred):
  return np.sqrt(np.mean((y_true - y_pred) ** 2))


def compute_mae(y_true, y_pred):
  return np.mean(np.abs(y_true - y_pred))


def compute_r2(y_true, y_pred):
  """Compute R-squared score"""
  res = np.sum((y_true - y_pred) ** 2)
  tot = np.sum((y_true - np.mean(y_true)) ** 2)
  return 1 - ( res / tot)

def print_metrics(y_true, y_pred, dataset=""):
  """Print all regression metrics for a dataset"""
  rmse = compute_rmse(y_true, y_pred)
  mae = compute_mae(y_true, y_pred)
  r2 = compute_r2(y_true, y_pred)

  print(f"{dataset} Metrics")
  print(f"RMSE: {rmse:.3f}")
  print(f"MAE: {mae:.3f}")
  print(f"R²: {r2:.3f}")
  return rmse, mae, r2

# Calculate predictions
y_train_pred = price_model.predict(X_train)
y_test_pred = price_model.predict(X_test)

# Compute and print metrics
train_rmse, train_mae, train_r2 = print_metrics(prices_train['Price'], y_train_pred, "Training")
test_rmse, test_mae, test_r2 = print_metrics(prices_test['Price'], y_test_pred, "Test")

# Visualize predictions vs actual
fig = px.scatter(
    x=prices_test['Price'],
    y=y_test_pred,
    title='Predicted vs Actual Prices',
    labels={'x': 'Actual Price', 'y': 'Predicted Price'}
)
fig.add_trace(px.line(x=[prices_test['Price'].min(), prices_test['Price'].max()], 
                      y=[prices_test['Price'].min(), prices_test['Price'].max()]).data[0])
fig.show()

Training Metrics
RMSE: 57.849
MAE: 40.027
R²: 0.763
Test Metrics
RMSE: 59.985
MAE: 41.118
R²: 0.736


### Problem 6c: MAE, MSE, and R-Squared

**Question:** What are the advantages of using MAE over MSE to measure a model's performance? What are the advantages of using MSE over MAE to measure a model's performance?

MAE treats all mistakes equally and is easy to understand, while MSE punishes big mistakes more and is better when large errors really matter.


<!-- END QUESTION -->

<!-- BEGIN QUESTION -->

**Question**: What does R-squared measure? Why is it significant?

R-squared measures how well your model explains the data, and it’s important because it shows how much of the outcome your model actually gets right.
